In [ ]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *

# Silver Data Transformation

In [ ]:
df = spark.read.format("delta")\
        .option("header",True)\
        .option("inferSchema",True)\
        .load("abfss://bronze@netflixprojectdlansh.dfs.core.windows.net/netflix_titles")

In [ ]:
df = df.fillna({"duration_minutes" : 0, "duration_seasons":1})

In [ ]:
df = df.withColumn("duration_minutes",col('duration_minutes').cast(IntegerType()))\
            .withColumn("duration_seasons",col('duration_seasons').cast(IntegerType()))

In [ ]:
df.printSchema()

In [ ]:
display(df)

In [ ]:
df = df.withColumn("Shorttitle",split(col('title'),':')[0])
df.display()

In [ ]:
df = df.withColumn("rating",split(col('rating'),'-')[0])
df.display()

In [ ]:
df = df.withColumn("type_flag",when(col('type')=='Movie',1)\
                        .when(col('type')=='TV Show',2)\
                        .otherwise(0))
display(df)

In [ ]:
from pyspark.sql.window import Window

In [ ]:
df = df.withColumn("duration_ranking",dense_rank().over(Window.orderBy(col('duration_minutes').desc())))

In [ ]:
df.display()

In [ ]:
df.createOrReplaceTempView("temp_view")

In [ ]:
df.createOrReplaceGlobalTempView("global_view")

In [ ]:
df = spark.sql(""" 
                select * from global_temp.global_view
               
               """)

In [ ]:
df.display()

In [ ]:
df_vis = df.groupBy("type").agg(count("*").alias("total_count"))
display(df_vis)

In [ ]:
df.write.format("delta")\
        .mode("overwrite")\
        .option("path","abfss://silver@netflixprojectdlansh.dfs.core.windows.net/netflix_titles")\
        .save()

In [ ]:
%python
__backend_agg_display_orig = display
__backend_agg_dfs = []
def __backend_agg_display_new(df):
    __backend_agg_df_modules = ["pandas.core.frame", "databricks.koalas.frame", "pyspark.sql.dataframe", "pyspark.pandas.frame", "pyspark.sql.connect.dataframe"]
    if (type(df).__module__ in __backend_agg_df_modules and type(df).__name__ == 'DataFrame') or isinstance(df, list):
        __backend_agg_dfs.append(df)

display = __backend_agg_display_new

def __backend_agg_user_code_fn():
    import base64
    exec(base64.standard_b64decode("ZGYgPSBkZi5ncm91cEJ5KCJ0eXBlIikuYWdnKGNvdW50KCIqIikuYWxpYXMoInRvdGFsX2NvdW50IikpCmRpc3BsYXkoZGYp").decode())

try:
    # run user code
    __backend_agg_user_code_fn()

    #reset display function
    display = __backend_agg_display_orig

    if len(__backend_agg_dfs) > 0:
        # create a temp view
        if type(__backend_agg_dfs[0]).__module__ == "databricks.koalas.frame":
            # koalas dataframe
            __backend_agg_dfs[0].to_spark().createOrReplaceTempView("DatabricksView4a1a870")
        elif type(__backend_agg_dfs[0]).__module__ == "pandas.core.frame" or isinstance(__backend_agg_dfs[0], list):
            # pandas dataframe
            spark.createDataFrame(__backend_agg_dfs[0]).createOrReplaceTempView("DatabricksView4a1a870")
        else:
            __backend_agg_dfs[0].createOrReplaceTempView("DatabricksView4a1a870")
        #run backend agg
        display(spark.sql("""WITH q AS (select * from DatabricksView4a1a870) SELECT `type`,SUM(`total_count`) `column_a91218a4224` FROM q GROUP BY `type`"""))
    else:
        displayHTML("dataframe no longer exists. If you're using dataframe.display(), use display(dataframe) instead.")


finally:
    spark.sql("drop view if exists DatabricksView4a1a870")
    display = __backend_agg_display_orig
    del __backend_agg_display_new
    del __backend_agg_display_orig
    del __backend_agg_dfs
    del __backend_agg_user_code_fn

